# Module 1 — Federated Training Core
### Blockchain-Based Dynamic Trust Modeling for Federated Fraud Detection

| Cell | What it does |
|------|-------------|
| **1** | Install all dependencies + verify versions |
| **2** | Upload the 5 Python source files |
| **3** | Upload / connect your dataset CSV |
| **4** | IEEE-CIS merge helper (skip if not using IEEE-CIS) |
| **5** | Set all configuration parameters |
| **6** | Run preprocessing pipeline + show partition table |
| **7** | Run federated simulation |
| **8** | Plot training curves (F1, AUC-ROC, Recall, Precision, Accuracy) |
| **9** | Per-bank evaluation table (final round) |
| **10** | Confusion matrix |
| **11** | Health check — PASS/FAIL against minimum thresholds |
| **12** | Download all result files |

> **Before running:** Runtime → Change runtime type → **T4 GPU**

## Cell 1 — Install Dependencies

In [ ]:
# Flower 1.7.0 pinned — Strategy API changed in 1.8+
# imbalanced-learn required for SMOTE oversampling
!pip install -q flwr==1.7.0 imbalanced-learn==0.11.0

import sys

# Clear any stale cached versions before reimporting
for _mod in list(sys.modules.keys()):
    if _mod.startswith(('flwr', 'imblearn')):
        del sys.modules[_mod]

import torch
import flwr
import sklearn
import imblearn
import numpy as np
import pandas as pd

print('=== Environment Check ===')
print(f'Python           : {sys.version.split()[0]}')
print(f'PyTorch          : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU              : {torch.cuda.get_device_name(0)}')
print(f'Flower (flwr)    : {flwr.__version__}')
print(f'scikit-learn     : {sklearn.__version__}')
print(f'imbalanced-learn : {imblearn.__version__}')
print(f'numpy            : {np.__version__}')
print(f'pandas           : {pd.__version__}')
print('=========================')

## Cell 2 — Upload Source Files

Upload all **5 Python files** from `split1_federated_core/`:
```
data_partition.py
local_models.py
flower_client.py
fedavg_strategy.py
main.py
```

In [ ]:
import sys
import os
from google.colab import files

print('Select all 5 .py files and upload together...')
uploaded = files.upload()

# Ensure /content is on the import path
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# Clear any previously imported versions of our modules
for _mod in list(sys.modules.keys()):
    if _mod in ('data_partition', 'local_models', 'flower_client',
                'fedavg_strategy', 'main'):
        del sys.modules[_mod]

REQUIRED = {
    'data_partition.py', 'local_models.py',
    'flower_client.py',  'fedavg_strategy.py', 'main.py'
}
missing = REQUIRED - set(uploaded.keys())

print('\nUpload status:')
for fname in sorted(REQUIRED):
    mark = '  OK' if fname in uploaded else '  MISSING'
    print(f'  {mark}  {fname}')

if missing:
    print(f'\nWARNING: {missing} not uploaded. Re-run this cell.')
else:
    print('\nAll 5 files uploaded successfully.')

## Cell 3 — Upload / Connect Dataset

Choose one option. Comment out the others. Leave `DATASET_PATH = None` to use synthetic data.

| Dataset | Label column | Size |
|---|---|---|
| Kaggle Credit Card Fraud | `Class` | 144 MB |
| IEEE-CIS Fraud (merged) | `isFraud` | ~900 MB |
| PaySim Synthetic | `isFraud` | ~470 MB |
| Synthetic (built-in) | auto | 0 MB |

In [ ]:
import os
import pandas as pd

DATASET_PATH = None    # <-- set this to use a real dataset

# ── OPTION A: Upload CSV directly (small/medium files) ───────────────────────
# from google.colab import files as colab_files
# print('Select your CSV file...')
# _up = colab_files.upload()
# DATASET_PATH = list(_up.keys())[0]

# ── OPTION B: Load from Google Drive (recommended for large files) ────────────
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = '/content/drive/MyDrive/datasets/creditcard.csv'

# ── OPTION C: Synthetic data — leave DATASET_PATH = None (default) ───────────

# ── Preview ───────────────────────────────────────────────────────────────────
if DATASET_PATH and os.path.exists(str(DATASET_PATH)):
    _preview = pd.read_csv(DATASET_PATH, nrows=3)
    _size_mb = os.path.getsize(DATASET_PATH) / 1e6
    print(f'File     : {DATASET_PATH}')
    print(f'Size     : {_size_mb:.1f} MB')
    print(f'Columns  : {list(_preview.columns)}')
    print()
    display(_preview)
else:
    print('DATASET_PATH not set -> will use SYNTHETIC data (10,000 samples, 3% fraud).')
    print('This is fine for testing. Use a real dataset for research results.')

## Cell 4 — IEEE-CIS Merge Helper *(skip if not using IEEE-CIS)*

IEEE-CIS ships as two files: `train_transaction.csv` + `train_identity.csv`.  
Run this cell once to merge them, then set `DATASET_PATH` in Cell 3.

In [ ]:
# Uncomment everything below if you are using the IEEE-CIS dataset.

# from google.colab import files as colab_files
# import pandas as pd
#
# print('Upload train_transaction.csv and train_identity.csv ...')
# colab_files.upload()   # select both files
#
# trans    = pd.read_csv('train_transaction.csv')
# identity = pd.read_csv('train_identity.csv')
# merged   = trans.merge(identity, on='TransactionID', how='left')
# merged.to_csv('ieee_cis_merged.csv', index=False)
#
# print(f'Merged shape : {merged.shape}')
# print(f'Fraud rate   : {merged["isFraud"].mean()*100:.3f}%')
# print('Saved as     : ieee_cis_merged.csv')
#
# # Then in Cell 3 set:
# # DATASET_PATH = 'ieee_cis_merged.csv'

print('Cell 4 — IEEE-CIS helper (commented out). Uncomment if needed.')

## Cell 5 — Configuration

Edit these values before running Cell 6.

In [ ]:
# =====================================================================
#  CONFIGURATION  — edit these values
# =====================================================================

NUM_CLIENTS   = 5       # Number of simulated bank nodes
NUM_ROUNDS    = 10      # Federated training rounds
MODEL_TYPE    = 'dnn'   # 'dnn' (recommended) or 'logistic'
ALPHA         = 0.5     # Dirichlet alpha: 0.1=very non-IID  5.0=near-IID
FRACTION_FIT  = 1.0     # Fraction of clients per round (1.0 = all)
USE_SMOTE     = True    # SMOTE oversampling on each bank's local data
LOG_DIR       = '/content/logs'

import os
os.makedirs(LOG_DIR, exist_ok=True)

print('=' * 48)
print('  MODULE 1 — CONFIGURATION')
print('=' * 48)
print(f'  Bank nodes      : {NUM_CLIENTS}')
print(f'  FL rounds       : {NUM_ROUNDS}')
print(f'  Local model     : {MODEL_TYPE}')
print(f'  Dirichlet alpha : {ALPHA}  ({"non-IID" if ALPHA < 1.0 else "near-IID"})')
print(f'  Clients/round   : {int(NUM_CLIENTS * FRACTION_FIT)} / {NUM_CLIENTS}')
print(f'  SMOTE           : {USE_SMOTE}')
print(f'  Dataset         : {DATASET_PATH if DATASET_PATH else "SYNTHETIC"}')
print(f'  Log directory   : {LOG_DIR}')
print('=' * 48)

## Cell 6 — Preprocessing + Partition

Runs the full 9-step pipeline:
1. Load CSV  2. Remove duplicates  3. Detect label  4. Enforce binary labels
5. Drop ID/string cols  6. Median imputation  7. Drop sparse cols
8. Winsorise outliers [1%, 99%]  9. StandardScaler

Then partitions data across bank nodes using Dirichlet distribution.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd

sys.path.insert(0, '/content')

# Clear stale module cache so re-uploaded files take effect
for _mod in list(sys.modules.keys()):
    if _mod in ('data_partition', 'local_models', 'flower_client',
                'fedavg_strategy', 'main'):
        del sys.modules[_mod]

from data_partition import load_dataset, dirichlet_partition, make_synthetic_data

# ── Load and preprocess ───────────────────────────────────────
if DATASET_PATH and os.path.exists(str(DATASET_PATH)):
    X, y = load_dataset(DATASET_PATH)
else:
    print('No dataset provided — using SYNTHETIC data.\n')
    X, y = make_synthetic_data(n_samples=10_000, n_features=30, fraud_rate=0.03)

# Store actual fraud rate so confusion matrix cell uses correct value later
ACTUAL_FRAUD_RATE = float(y.mean())

print(f'\nPreprocessed dataset:')
print(f'  Shape          : {X.shape}')
print(f'  Total samples  : {len(X):,}')
print(f'  Fraud samples  : {int(y.sum()):,}  ({ACTUAL_FRAUD_RATE*100:.3f}%)')
print(f'  Legit samples  : {int((y==0).sum()):,}')
print(f'  Feature dtype  : {X.dtype}')
print()

# ── Dirichlet partition ───────────────────────────────────────
partitions = dirichlet_partition(X, y, num_clients=NUM_CLIENTS, alpha=ALPHA)

# ── Show partition table ──────────────────────────────────────
rows = []
for p in partitions:
    rows.append({
        'Bank':           f"Bank {p['client_id']:02d}",
        'Train samples':  len(p['X_train']),
        'Test samples':   len(p['X_test']),
        'Fraud (train)':  int(p['y_train'].sum()),
        'Fraud rate':     f"{p['y_train'].mean()*100:.2f}%",
    })
print(f'\nPartition summary — {NUM_CLIENTS} bank nodes (Dirichlet alpha={ALPHA}):')
display(pd.DataFrame(rows).set_index('Bank'))

## Cell 7 — Run Federated Simulation

In [ ]:
import flwr as fl
from flower_client   import make_client_fn
from fedavg_strategy import get_fedavg_strategy

print('=' * 62)
print('  MODULE 1 — FEDERATED TRAINING')
print(f'  Model: {MODEL_TYPE.upper()}  |  Banks: {NUM_CLIENTS}  |  Rounds: {NUM_ROUNDS}')
print('=' * 62)
print()

# Client factory — creates one BankFederatedClient per bank node
client_fn = make_client_fn(
    partitions=partitions,
    model_type=MODEL_TYPE,
    use_smote=USE_SMOTE,
)

# FedAvg strategy — aggregates updates, logs metrics each round
strategy = get_fedavg_strategy(
    num_clients=NUM_CLIENTS,
    fraction_fit=FRACTION_FIT,
    log_dir=LOG_DIR,
)

# Run simulation — all bank nodes run as threads on this machine
fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy,
    client_resources={'num_cpus': 1, 'num_gpus': 0.0},
)

strategy.print_summary()
print('Logs written to:', LOG_DIR)

## Cell 8 — Training Curves

Plots F1, AUC-ROC, Recall, Precision, Accuracy across all rounds.  
A dashed vertical line marks the best round for each metric.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

log_path = f'{LOG_DIR}/training_log.json'
with open(log_path) as f:
    logs = json.load(f)

rounds  = [l['round']            for l in logs]
f1s     = [l['global_f1']        for l in logs]
aucs    = [l['global_auc']       for l in logs]
recalls = [l['global_recall']    for l in logs]
precs   = [l['global_precision'] for l in logs]
accs    = [l['global_accuracy']  for l in logs]
losses  = [l['global_loss']      for l in logs]

# ── 5-panel metric plot ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
_ds_label = DATASET_PATH.split('/')[-1] if DATASET_PATH else 'synthetic'
fig.suptitle(
    f'Module 1 — FedAvg Baseline  |  {NUM_CLIENTS} banks  '
    f'|  {NUM_ROUNDS} rounds  |  {_ds_label}',
    fontsize=12, fontweight='bold'
)

panels = [
    (f1s,     'Global F1 Score',  '#2196F3'),
    (aucs,    'Global AUC-ROC',   '#4CAF50'),
    (recalls, 'Global Recall',    '#FF9800'),
    (precs,   'Global Precision', '#9C27B0'),
    (accs,    'Global Accuracy',  '#F44336'),
]

for ax, (vals, title, color) in zip(axes, panels):
    ax.plot(rounds, vals, 'o-', linewidth=2, color=color, markersize=5)
    ax.fill_between(rounds, vals, alpha=0.10, color=color)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Federation Round')
    ax.set_ylabel('Score')
    ax.set_ylim(0.0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))
    best_idx = int(np.argmax(vals))
    ax.axvline(x=rounds[best_idx], color=color, linestyle='--', alpha=0.5)
    ax.annotate(
        f'best\n{vals[best_idx]:.3f}',
        xy=(rounds[best_idx], vals[best_idx]),
        xytext=(5, -22), textcoords='offset points',
        fontsize=7, color=color
    )

plt.tight_layout()
plot_path = f'{LOG_DIR}/training_curves.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')

# ── Loss curve ──────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(7, 3))
ax2.plot(rounds, losses, 's-', color='#607D8B', linewidth=2, markersize=5)
ax2.set_title('Global Loss (1 - F1) per Round', fontweight='bold')
ax2.set_xlabel('Round')
ax2.set_ylabel('Loss')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
loss_path = f'{LOG_DIR}/loss_curve.png'
plt.savefig(loss_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {loss_path}')

## Cell 9 — Per-Bank Evaluation Table

Every bank node's individual F1, AUC-ROC, Recall, Precision, Accuracy for the **final round**.  
Also shows the global F1 progress bar across all rounds.

In [ ]:
import json
import pandas as pd

log_path = f'{LOG_DIR}/training_log.json'
with open(log_path) as f:
    logs = json.load(f)

final = logs[-1]

# ── Global metrics summary ─────────────────────────────────────────
print('=' * 58)
print('  FINAL ROUND — GLOBAL EVALUATION')
print('=' * 58)
print(f'  Round          : {final["round"]}')
print(f'  Global F1      : {final["global_f1"]:.4f}')
print(f'  Global AUC-ROC : {final["global_auc"]:.4f}')
print(f'  Global Recall  : {final["global_recall"]:.4f}')
print(f'  Global Prec.   : {final["global_precision"]:.4f}')
print(f'  Global Accuracy: {final["global_accuracy"]:.4f}')
print(f'  Loss (1-F1)    : {final["global_loss"]:.4f}')
print(f'  Best F1 ever   : {final["best_f1"]:.4f}  (round {final["best_round"]})')
print('=' * 58)

# ── Per-bank table ─────────────────────────────────────────────────
df_clients = pd.DataFrame(final['client_metrics'])
df_clients['client_id'] = df_clients['client_id'].astype(int)
df_clients = df_clients.rename(columns={
    'client_id': 'Bank',
    'f1':        'F1',
    'auc_roc':   'AUC-ROC',
    'recall':    'Recall',
    'precision': 'Precision',
    'accuracy':  'Accuracy',
})
df_clients['Bank'] = df_clients['Bank'].apply(lambda x: f'Bank {x:02d}')
df_clients = df_clients.set_index('Bank').round(4)

print('\nPer-Bank Metrics — Final Round:')
display(
    df_clients.style
        .background_gradient(subset=['F1', 'AUC-ROC', 'Recall'], cmap='Greens')
        .format('{:.4f}')
)

# ── Round-by-round F1 progress bar ────────────────────────────────
print('\nGlobal F1 — all rounds:')
for l in logs:
    filled = int(l['global_f1'] * 40)
    bar    = chr(9608) * filled + chr(9617) * (40 - filled)
    tag    = ' <- BEST' if l['round'] == final['best_round'] else ''
    print(f"  Round {l['round']:02d} | {bar} {l['global_f1']:.4f}{tag}")

## Cell 10 — Confusion Matrix

Reconstructed from the aggregated Precision and Recall of the final round.  
Uses the **actual fraud rate from your dataset** (not a hardcoded guess).

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

log_path = f'{LOG_DIR}/training_log.json'
with open(log_path) as f:
    logs = json.load(f)
final = logs[-1]

# ── Reconstruct confusion matrix from aggregated metrics ──────────
# Uses actual fraud rate (ACTUAL_FRAUD_RATE) computed in Cell 6.
# Formula:
#   actual_positives = n_test * fraud_rate
#   TP = recall  * actual_positives
#   FN = actual_positives - TP
#   FP = TP / precision - TP
#   TN = n_test - TP - FN - FP

precision = final['global_precision']
recall    = final['global_recall']
n_test    = sum(len(p['X_test']) for p in partitions)

# ACTUAL_FRAUD_RATE was stored in Cell 6 from the real data
_fraud_rate = ACTUAL_FRAUD_RATE

if precision > 0 and recall > 0:
    actual_pos = max(1, int(n_test * _fraud_rate))
    TP = int(recall * actual_pos)
    FN = actual_pos - TP
    FP = max(0, int(TP / precision) - TP)
    TN = max(0, n_test - TP - FN - FP)   # max(0,...) prevents negative TN

    cm = np.array([[TN, FP], [FN, TP]])

    # ── Plot ─────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax)

    cell_labels = [
        ['TN\n(Legit correctly\nidentified)', 'FP\n(Legit flagged\nas fraud)'],
        ['FN\n(Fraud missed\n** critical **)', 'TP\n(Fraud correctly\ncaught)']
    ]
    cm_max = cm.max() if cm.max() > 0 else 1

    for row in range(2):
        for col in range(2):
            val = cm[row, col]
            text_color = 'white' if val > cm_max * 0.6 else 'black'
            ax.text(
                col, row,
                f'{val:,}\n{cell_labels[row][col]}',
                ha='center', va='center',
                fontsize=9, color=text_color
            )

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Predicted: Legit', 'Predicted: Fraud'], fontsize=10)
    ax.set_yticklabels(['Actual: Legit', 'Actual: Fraud'], fontsize=10)
    ax.set_title(
        f'Confusion Matrix (estimated from final round metrics)\n'
        f'F1={final["global_f1"]:.4f}  |  n_test={n_test:,}  '
        f'|  fraud_rate={_fraud_rate*100:.2f}%',
        fontweight='bold', fontsize=10
    )
    plt.tight_layout()
    cm_path = f'{LOG_DIR}/confusion_matrix.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {cm_path}')

    # ── Numeric breakdown ────────────────────────────────────────
    print(f'\nEstimated confusion matrix values:')
    print(f'  TN (legit correctly identified) : {TN:,}')
    print(f'  FP (legit wrongly flagged)       : {FP:,}')
    print(f'  FN (fraud missed) [CRITICAL]     : {FN:,}')
    print(f'  TP (fraud correctly caught)      : {TP:,}')
    print(f'\n  Catch rate (Recall)  : {recall:.4f}  ({TP}/{actual_pos} fraud caught)')
    print(f'  False alarm rate     : {FP/(FP+TN):.4f}  ({FP} legit wrongly flagged)')
else:
    print('WARNING: precision or recall is zero.')
    print('The model may not have converged yet. Try more rounds.')

## Cell 11 — Health Check: PASS / FAIL

Tells you whether Module 1 results are good enough to proceed to Module 2.

### Expected score ranges by dataset

| Dataset | F1 | AUC-ROC | Recall |
|---|---|---|---|
| Kaggle Credit Card Fraud | 0.75 – 0.92 | 0.95 – 0.99 | 0.80 – 0.95 |
| IEEE-CIS Fraud Detection | 0.60 – 0.80 | 0.88 – 0.95 | 0.65 – 0.85 |
| PaySim Synthetic | 0.70 – 0.90 | 0.94 – 0.99 | 0.75 – 0.92 |
| Synthetic (built-in test) | 0.45 – 0.75 | 0.70 – 0.90 | 0.45 – 0.80 |

> **Why F1 and Recall matter more than Accuracy:**  
> A model predicting everything as 'legit' achieves 99.8% accuracy on Kaggle Credit Card Fraud  
> but 0% Recall — it catches zero fraud. Always judge by F1 and Recall first.

In [ ]:
import json
import pandas as pd

# ── Thresholds — adjust per dataset ──────────────────────────────────────────
THRESHOLD_F1        = 0.50   # minimum F1
THRESHOLD_AUC       = 0.75   # minimum AUC-ROC
THRESHOLD_RECALL    = 0.50   # minimum Recall (fraud catch rate)
THRESHOLD_PRECISION = 0.30   # minimum Precision

log_path = f'{LOG_DIR}/training_log.json'
with open(log_path) as f:
    logs = json.load(f)

# Use best F1 round for the pass/fail check
best_log  = max(logs, key=lambda l: l['global_f1'])
final_log = logs[-1]

f1_val    = best_log['global_f1']
auc_val   = best_log['global_auc']
rec_val   = best_log['global_recall']
pre_val   = best_log['global_precision']
acc_val   = best_log['global_accuracy']

def _check(name, value, threshold):
    passed  = value >= threshold
    status  = 'PASS' if passed else 'FAIL'
    gap     = value - threshold
    gap_str = f'+{gap:.4f}' if gap >= 0 else f'{gap:.4f}'
    print(f'  [{status}]  {name:<18} {value:.4f}  '
          f'(threshold >= {threshold:.2f},  gap {gap_str})')
    return passed

print('=' * 65)
print(f'  MODULE 1 HEALTH CHECK  (best round = {best_log["round"]})')
print('=' * 65)
results = [
    _check('F1 Score',   f1_val,  THRESHOLD_F1),
    _check('AUC-ROC',    auc_val, THRESHOLD_AUC),
    _check('Recall',     rec_val, THRESHOLD_RECALL),
    _check('Precision',  pre_val, THRESHOLD_PRECISION),
]
print(f'  [INFO]  Accuracy         {acc_val:.4f}  (informational only)')
print('=' * 65)

all_pass = all(results)
if all_pass:
    print('  ALL CHECKS PASSED')
    print('  Module 1 is working correctly.')
    print('  You can proceed to Module 2 (trust-weighted aggregation).')
else:
    n_fail = results.count(False)
    print(f'  {n_fail} check(s) FAILED. Suggestions:')
    if not results[0]:
        print('    F1 low    -> increase NUM_ROUNDS, enable USE_SMOTE=True')
    if not results[1]:
        print('    AUC low   -> model not separating classes; try MODEL_TYPE="dnn"')
    if not results[2]:
        print('    Recall low -> DNN missing fraud; increase pos_weight in local_models.py')
    if not results[3]:
        print('    Precision  -> too many false positives; acceptable for fraud detection')
print('=' * 65)

# ── Full round summary table ──────────────────────────────────────────────────
print('\nFull round-by-round results:')
summary_df = pd.DataFrame([{
    'Round':     l['round'],
    'F1':        round(l['global_f1'],        4),
    'AUC-ROC':   round(l['global_auc'],       4),
    'Recall':    round(l['global_recall'],    4),
    'Precision': round(l['global_precision'], 4),
    'Accuracy':  round(l['global_accuracy'],  4),
    'Loss':      round(l['global_loss'],      4),
} for l in logs]).set_index('Round')

display(
    summary_df.style
        .highlight_max(subset=['F1', 'AUC-ROC', 'Recall'], color='lightgreen')
        .highlight_min(subset=['Loss'], color='lightblue')
        .format('{:.4f}')
)

## Cell 12 — Download Results

In [ ]:
import os
from google.colab import files

to_download = [
    f'{LOG_DIR}/training_log.json',
    f'{LOG_DIR}/training_curves.png',
    f'{LOG_DIR}/loss_curve.png',
    f'{LOG_DIR}/confusion_matrix.png',
]

print('Downloading result files...')
for path in to_download:
    fname = path.split('/')[-1]
    if os.path.exists(path):
        files.download(path)
        print(f'  OK   {fname}')
    else:
        print(f'  SKIP {fname}  (file not found — run earlier cells first)')

print('\nDone. Files will appear in your browser downloads.')